In [1]:
import os
from pyspark.sql import SparkSession

#os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home"
#os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

spark = SparkSession.builder \
    .appName("GarbageStats") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/26 19:44:27 WARN Utils: Your hostname, WIN-BTCF9G469IC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/26 19:44:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 19:44:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [41]:
df_train = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load("../data/train")

df_test = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load("../data/test")

In [42]:
from pyspark.sql.functions import lit, element_at, size, count, when, col, min, max, avg, split

In [43]:
df_train = df_train.withColumn("split", lit("train"))
df_test = df_test.withColumn("split", lit("test"))

df_raw = df_train.unionByName(df_test)

df_raw = df_raw.withColumn(
    "category",
    element_at(split("path", "/"), -2)
)

In [44]:
df_raw.count()

10464

In [45]:
df_raw.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]).show()

+----+----------------+------+-------+-----+--------+
|path|modificationTime|length|content|split|category|
+----+----------------+------+-------+-----+--------+
|   0|               0|     0|      0|    0|       0|
+----+----------------+------+-------+-----+--------+



In [46]:
df_raw.groupBy("split").count().show()

+-----+-----+
|split|count|
+-----+-----+
|train| 7324|
| test| 3140|
+-----+-----+



In [47]:
df_raw.groupBy("category").count().orderBy("count", ascending=False).show()

+-------------+-----+
|     category|count|
+-------------+-----+
|        glass| 2649|
|biodegradable| 2220|
|        paper| 1746|
|    cardboard| 1439|
|        metal| 1248|
|      plastic| 1162|
+-------------+-----+



In [48]:
df_raw.select("length").describe().show()

+-------+------------------+
|summary|            length|
+-------+------------------+
|  count|             10464|
|   mean|19151.321769877675|
| stddev| 9474.638532461628|
|    min|              4188|
|    max|             67006|
+-------+------------------+



In [122]:
df_raw.groupBy("path").count().filter("count > 1").show() #doublons

+----+-----+
|path|count|
+----+-----+
+----+-----+



# DATA Transformées

In [50]:
df_train = spark.read.parquet("../data/train_data.parquet")
df_test = spark.read.parquet("../data/test_data.parquet")

In [89]:
df_train.printSchema()
df_test.printSchema()

root
 |-- x_train: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- y_train: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- class: string (nullable = true)
 |-- height: integer (nullable = true)
 |-- width: integer (nullable = true)

root
 |-- x_test: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: integer (containsNull = true)
 |-- y_test: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- class: string (nullable = true)
 |-- height: integer (nullable = true)
 |-- width: integer (nullable = true)



In [53]:
df_train.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_train.columns
]).show()

+-------+-------+-----+
|x_train|y_train|class|
+-------+-------+-----+
|      0|      0|    0|
+-------+-------+-----+



In [54]:
df_test.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_test.columns
]).show()

+------+------+-----+
|x_test|y_test|class|
+------+------+-----+
|     0|     0|    0|
+------+------+-----+



In [55]:
df_train.groupBy("class").count().orderBy("count", ascending=False).show()
df_test.groupBy("class").count().orderBy("count", ascending=False).show()

+-------------+-----+
|        class|count|
+-------------+-----+
|        glass| 1854|
|biodegradable| 1554|
|        paper| 1223|
|    cardboard| 1006|
|        metal|  874|
|      plastic|  813|
+-------------+-----+

+-------------+-----+
|        class|count|
+-------------+-----+
|        glass|  795|
|biodegradable|  666|
|        paper|  523|
|    cardboard|  433|
|        metal|  374|
|      plastic|  349|
+-------------+-----+



In [56]:
df_train = df_train.withColumn("height", size("x_train"))
df_train = df_train.withColumn("width", size(col("x_train")[0]))

df_train.groupBy("height", "width").count().show()

+------+-----+-----+
|height|width|count|
+------+-----+-----+
|    64|   64| 7324|
+------+-----+-----+



In [57]:
df_test = df_test.withColumn("height", size("x_test"))
df_test = df_test.withColumn("width", size(col("x_test")[0]))

df_test.groupBy("height", "width").count().show()

+------+-----+-----+
|height|width|count|
+------+-----+-----+
|    64|   64| 3140|
+------+-----+-----+



In [58]:
df_train.withColumn("label_size", size("y_train")) \
    .groupBy("label_size") \
    .count() \
    .show()

+----------+-----+
|label_size|count|
+----------+-----+
|         6| 7324|
+----------+-----+



In [60]:
df_test.withColumn("label_size", size("y_test")) \
    .groupBy("label_size") \
    .count() \
    .show()

+----------+-----+
|label_size|count|
+----------+-----+
|         6| 3140|
+----------+-----+



# Prédiction DATA

In [110]:
df_pred = spark.read.parquet("../data/prediction_data.parquet")

In [111]:
df_pred.count()

476

In [112]:
df_pred.groupBy("class").count().orderBy("count", ascending=False).show()

+-------------+-----+
|        class|count|
+-------------+-----+
|        paper|  112|
|        glass|  112|
|      plastic|  108|
|        metal|   90|
|    cardboard|   27|
|biodegradable|   27|
+-------------+-----+



In [113]:
df_pred.groupBy("file").count().filter("count > 1").show()

+-------------+-----+
|         file|count|
+-------------+-----+
|plastic36.jpg|    2|
|plastic35.jpg|    2|
|plastic29.jpg|    2|
|plastic12.jpg|    2|
|plastic27.jpg|    2|
|plastic32.jpg|    2|
|plastic31.jpg|    2|
|plastic26.jpg|    2|
|plastic18.jpg|    2|
|plastic13.jpg|    2|
|plastic30.jpg|    2|
|plastic22.jpg|    2|
|plastic21.jpg|    2|
|plastic33.jpg|    2|
|plastic25.jpg|    2|
|plastic16.jpg|    2|
|plastic38.jpg|    2|
|plastic14.jpg|    2|
|plastic44.jpg|    2|
|plastic28.jpg|    2|
+-------------+-----+
only showing top 20 rows


In [114]:
df_pred.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_pred.columns
]).show()

+----+----------+--------+-----+----------+
|file|prediction|class_id|class|confidence|
+----+----------+--------+-----+----------+
|   0|         0|       0|    0|         0|
+----+----------+--------+-----+----------+



In [115]:
total = df_pred.count()

df_pred.groupBy("class") \
    .count() \
    .withColumn("percentage", col("count") / total * 100) \
    .orderBy("percentage", ascending=False) \
    .show()

+-------------+-----+------------------+
|        class|count|        percentage|
+-------------+-----+------------------+
|        paper|  112| 23.52941176470588|
|        glass|  112| 23.52941176470588|
|      plastic|  108|22.689075630252102|
|        metal|   90|18.907563025210084|
|    cardboard|   27|5.6722689075630255|
|biodegradable|   27|5.6722689075630255|
+-------------+-----+------------------+



In [116]:
df_pred.groupBy("class") \
    .agg(avg("confidence").alias("avg_confidence")) \
    .orderBy("avg_confidence", ascending=False) \
    .show()

+-------------+-------------------+
|        class|     avg_confidence|
+-------------+-------------------+
|        paper| 0.5859507456687945|
|      plastic| 0.5615507062110636|
|    cardboard| 0.5137471755345663|
|        glass| 0.4502898883074522|
|        metal|0.36566069225470227|
|biodegradable|0.28432592197700784|
+-------------+-------------------+



In [117]:
df_pred.filter(col("confidence") < 0.5).show()

+-------------+--------------------+--------+-------------+-------------------+
|         file|          prediction|class_id|        class|         confidence|
+-------------+--------------------+--------+-------------+-------------------+
|  paper79.jpg|[0.05858080089092...|       5|        paper| 0.4362509250640869|
|  glass75.jpg|[0.01232929807156...|       3|        glass| 0.4934532642364502|
|  glass12.jpg|[0.16704802215099...|       4|        metal| 0.2654241919517517|
|plastic29.jpg|[0.30113026499748...|       5|        paper| 0.3295934498310089|
|  paper34.jpg|[0.10291779786348...|       5|        paper| 0.2711322605609894|
|  paper21.jpg|[0.01727584004402...|       3|        glass|0.36399200558662415|
|   glass7.jpg|[0.00376888969913...|       6|      plastic|0.46028026938438416|
|  glass23.jpg|[0.01992154866456...|       6|      plastic| 0.3557801842689514|
|plastic12.jpg|[0.01694130338728...|       5|        paper|0.39191409945487976|
|  paper26.jpg|[0.12414984405040...|    

In [118]:
df_pred.select("confidence").describe().show()

+-------+-------------------+
|summary|         confidence|
+-------+-------------------+
|  count|                476|
|   mean|0.48563836809216426|
| stddev|0.22225839547900592|
|    min|0.17791776359081268|
|    max|  0.999238133430481|
+-------+-------------------+



In [119]:
df_pred.orderBy("confidence", ascending=False).show(10)

+--------------+--------------------+--------+---------+------------------+
|          file|          prediction|class_id|    class|        confidence|
+--------------+--------------------+--------+---------+------------------+
|plastic313.jpg|[7.41114472308651...|       2|cardboard| 0.999238133430481|
|   glass59.jpg|[1.56333559431232...|       6|  plastic| 0.995294988155365|
|   metal85.jpg|[2.11534315894823...|       4|    metal|0.9940873384475708|
|   glass17.jpg|[6.47840465717308...|       6|  plastic|0.9936113953590393|
|plastic330.jpg|[1.37115057441405...|       6|  plastic|0.9916015267372131|
|   paper97.jpg|[7.18519950169138...|       5|    paper|0.9915831685066223|
|   paper93.jpg|[4.18072631873656...|       5|    paper|0.9903488159179688|
|  paper120.jpg|[1.87678997463081...|       5|    paper| 0.989693820476532|
|  paper127.jpg|[6.01191190071404...|       5|    paper|0.9837069511413574|
|  paper123.jpg|[2.30466303037246...|       5|    paper|0.9835513234138489|
+-----------

# Ecriture parquet 

In [120]:
df_pred.select(count("*").alias("total")) \
    .write.mode("overwrite") \
    .parquet("../data/stats/total.parquet")

In [121]:
df_pred.groupBy("class") \
    .count() \
    .orderBy("count", ascending=False) \
    .write.mode("overwrite") \
    .parquet("../data/stats/count_by_class.parquet")

In [103]:
df_pred.groupBy("file") \
    .count() \
    .filter("count > 1") \
    .write.mode("overwrite") \
    .parquet("../data/stats/duplicates.parquet")

In [123]:
df_pred.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_pred.columns
]).write.mode("overwrite") \
 .parquet("../data/stats/nulls.parquet")

In [124]:
total = df_pred.count()

df_pred.groupBy("class") \
    .count() \
    .withColumn("percentage", col("count") / total * 100) \
    .orderBy("percentage", ascending=False) \
    .write.mode("overwrite") \
    .parquet("../data/stats/percentage_by_class.parquet")

In [125]:
df_pred.groupBy("class") \
    .agg(avg("confidence").alias("avg_confidence")) \
    .orderBy("avg_confidence", ascending=False) \
    .write.mode("overwrite") \
    .parquet("../data/stats/confidence_by_class.parquet")

In [130]:
df_pred.filter(col("confidence") < 0.5) \
    .write.mode("overwrite") \
    .parquet("../data/stats/low_confidence.parquet")

In [129]:
train_count = df_train.count()
test_count = df_test.count()

total = train_count + test_count

print("Train :", train_count)
print("Test :", test_count)
print("Total :", total)

Train : 7324
Test : 3140
Total : 10464


In [127]:
df_pred.select("confidence") \
    .describe() \
    .write.mode("overwrite") \
    .parquet("../data/stats/confidence_stats.parquet")

In [128]:
df_pred.orderBy("confidence", ascending=False) \
    .limit(10) \
    .write.mode("overwrite") \
    .parquet("../data/stats/top_predictions.parquet")